# 01 · Generación de la fuente de datos — DataRetail 360 (EP3)

**IDY1102 · Estructuras y Visualización de Datos · Evaluación Parcial 3**

| Campo | Detalle |
|---|---|
| **Propósito** | Generar la fuente de datos del caso sobre la que trabaja el pipeline ETL y el modelo dimensional |
| **Base** | Generador de ventas del notebook **2.3.5 Pipeline robusto** (mismo catálogo, sucursales, canales y problemas de calidad) |
| **Extensión del equipo** | Maestros de productos, sucursales, clientes y campañas · boletas con número de línea · precio, descuento y costo · segunda entrega de maestros con cambios (insumo de SCD 1, 2 y 3) · inventario diario por producto y sucursal |
| **Reproducibilidad** | Semilla fija + `manifest.json` con parámetros, conteos y huella SHA-256 de cada archivo |

### Por qué se extiende la fuente del 2.3.5
La fuente del curso trae solo `id_venta, fecha, sku, sucursal, canal, cantidad, monto, estado, rut_cliente`. Con eso no es posible construir:
- una **dimensión espacial** (no hay comuna, región ni coordenadas de sucursal),
- **jerarquías** completas de producto (no hay subcategoría ni marca),
- la dimensión **Promoción/Campaña** ni el **descuento**,
- la **dimensión degenerada** boleta/línea,
- el **margen** (no hay costo),
- ni demostrar **SCD**, porque los maestros no cambian en el tiempo.

Por eso se enriquece con **datos maestros sintéticos** coherentes con las ventas. Todos los datos personales (RUT, nombres, correos) son **ficticios**.


## 0. Parámetros
Ejecutar este notebook desde la carpeta `notebooks/` del proyecto. Los archivos se escriben en `data/fuente/`.

In [ ]:
from pathlib import Path
import json, hashlib, shutil, time, unicodedata
import numpy as np
import pandas as pd

N_LINEAS        = 500_000        # líneas de venta aprox. (bajar a 50_000 para una prueba rápida)
DIAS            = 90             # 1 archivo CSV por día: 2025-01-01 … 2025-03-31
FECHA_INICIO    = "2025-01-01"
FECHA_MAESTROS2 = "2025-03-01"   # fecha en que entra en vigencia la 2.ª entrega de maestros (v2)
N_CLIENTES      = 30_000         # clientes de la 1.ª entrega
N_CLIENTES_NUEVOS = 1_500        # clientes que se registran en la 2.ª entrega
SEMILLA         = 42

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FUENTE   = BASE / "data" / "fuente"
DIR_VENT = FUENTE / "ventas"
DIR_M1   = FUENTE / "maestros" / "v1"
DIR_M2   = FUENTE / "maestros" / "v2"
DIR_INV  = FUENTE / "inventario"
if FUENTE.exists():
    shutil.rmtree(FUENTE)                     # regenerar desde cero (reproducible por semilla)
for d in (DIR_VENT, DIR_M1, DIR_M2, DIR_INV):
    d.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEMILLA)
print("pandas", pd.__version__, "| numpy", np.__version__, "| fuente en", FUENTE)
 # error de ".venv aun con atados"

: 

## 1. Maestro de productos (v1 y v2)
Mismos 16 SKU y precios del notebook 2.3.5, agregando **subcategoría, marca y costo unitario** para completar la jerarquía *Producto → Subcategoría → Categoría* y calcular margen.

**Cambios de la v2 (vigentes desde `FECHA_MAESTROS2`)** y el tipo de SCD que motivan:

| SKU | Cambio | Naturaleza del cambio | SCD sugerido |
|---|---|---|---|
| NB-001, CE-004, CO-015 | Cambia `precio_lista` | Cambio real de negocio: las ventas deben valorizarse con el precio vigente | Tipo 2 |
| SW-011 | `categoria` Tecnología → Accesorios | Reclasificación: los reportes históricos no deben cambiar | Tipo 2 |
| MS-008 | `marca` "KUMBRE " → "Kumbre" | Corrección de un error de digitación | Tipo 1 |
| GA-017 | Producto nuevo | Alta en la dimensión | Inserción |

In [2]:
COLS_PROD = ["sku", "producto", "subcategoria", "categoria", "marca", "precio_lista", "costo_unitario"]
productos_v1 = pd.DataFrame([
    ("NB-001", "Notebook 14",   "Computación",    "Tecnología",   "Andes Tech", 799_000, 600_000),
    ("AU-002", "Audífonos",     "Audio",          "Accesorios",   "Sonora",      30_000,  16_000),
    ("TV-003", "Smart TV 55",   "Televisión",     "Tecnología",   "Pacífico",   459_000, 350_000),
    ("CE-004", "Celular",       "Telefonía",      "Tecnología",   "Nova",       349_000, 255_000),
    ("TB-005", "Tablet 10",     "Computación",    "Tecnología",   "Nova",       229_000, 165_000),
    ("MO-006", "Monitor 27",    "Computación",    "Tecnología",   "Andes Tech", 189_000, 132_000),
    ("TE-007", "Teclado",       "Periféricos",    "Accesorios",   "Kumbre",      25_000,  12_500),
    ("MS-008", "Mouse",         "Periféricos",    "Accesorios",   "KUMBRE ",     15_000,   7_000),
    ("IM-009", "Impresora",     "Impresión",      "Tecnología",   "Pacífico",   129_000,  92_000),
    ("PA-010", "Parlante",      "Audio",          "Accesorios",   "Sonora",      59_000,  33_000),
    ("SW-011", "Smartwatch",    "Wearables",      "Tecnología",   "Nova",       149_000,  98_000),
    ("CA-012", "Cámara",        "Fotografía",     "Tecnología",   "Pacífico",   399_000, 290_000),
    ("RO-013", "Router WiFi",   "Redes",          "Accesorios",   "Kumbre",      45_000,  26_000),
    ("DI-014", "Disco SSD 1TB", "Almacenamiento", "Accesorios",   "Andes Tech",  69_000,  44_000),
    ("CO-015", "Consola",       "Consolas",       "Gaming",       "Volcán",     499_000, 395_000),
    ("MI-016", "Microondas",    "Cocina",         "Electrohogar", "Pacífico",    89_000,  61_000),
], columns=COLS_PROD)

productos_v2 = productos_v1.copy()
cambios_precio = {"NB-001": 749_000, "CE-004": 329_000, "CO-015": 529_000}
for sku, precio in cambios_precio.items():
    productos_v2.loc[productos_v2["sku"] == sku, "precio_lista"] = precio
productos_v2.loc[productos_v2["sku"] == "SW-011", "categoria"] = "Accesorios"
productos_v2.loc[productos_v2["sku"] == "MS-008", "marca"] = "Kumbre"
productos_v2 = pd.concat([productos_v2, pd.DataFrame(
    [("GA-017", "Audífonos Gamer", "Audio Gamer", "Gaming", "Volcán", 69_000, 41_000)], columns=COLS_PROD)],
    ignore_index=True)

def escribir_json(df, ruta):
    ruta.write_text(json.dumps(df.to_dict("records"), ensure_ascii=False, indent=1, default=int), encoding="utf-8")

escribir_json(productos_v1, DIR_M1 / "productos.json")
escribir_json(productos_v2, DIR_M2 / "productos.json")
productos_v2

,sku,producto,subcategoria,categoria,marca,precio_lista,costo_unitario
0,NB-001,Notebook 14,Computación,Tecnología,Andes Tech,749000,600000
1,AU-002,Audífonos,Audio,Accesorios,Sonora,30000,16000
2,TV-003,Smart TV 55,Televisión,Tecnología,Pacífico,459000,350000
3,CE-004,Celular,Telefonía,Tecnología,Nova,329000,255000
4,TB-005,Tablet 10,Computación,Tecnología,Nova,229000,165000
5,MO-006,Monitor 27,Computación,Tecnología,Andes Tech,189000,132000
6,TE-007,Teclado,Periféricos,Accesorios,Kumbre,25000,12500
7,MS-008,Mouse,Periféricos,Accesorios,Kumbre,15000,7000
8,IM-009,Impresora,Impresión,Tecnología,Pacífico,129000,92000
9,PA-010,Parlante,Audio,Accesorios,Sonora,59000,33000


## 2. Maestro de sucursales (v1 y v2)
Las 8 sucursales del 2.3.5 con **comuna, región, coordenadas** (dimensión espacial) y **formato de tienda**.

En la v2 dos sucursales cambian de formato por remodelación. Al negocio le interesa comparar *formato actual vs. formato anterior* (antes/después de la remodelación), pero no toda la historia → **SCD Tipo 3** (`formato_actual` + `formato_anterior`).

In [3]:
sucursales_v1 = pd.DataFrame([
    ("S01", "Providencia",  "Tienda grande",  "Providencia",  "Metropolitana", -33.4255, -70.6114, "2015-03-10"),
    ("S02", "Las Condes",   "Tienda grande",  "Las Condes",   "Metropolitana", -33.4170, -70.5500, "2016-08-22"),
    ("S03", "Maipú",        "Tienda mediana", "Maipú",        "Metropolitana", -33.5110, -70.7580, "2018-05-14"),
    ("S04", "Ñuñoa",        "Tienda mediana", "Ñuñoa",        "Metropolitana", -33.4569, -70.5973, "2019-11-04"),
    ("S05", "Concepción",   "Tienda grande",  "Concepción",   "Biobío",        -36.8270, -73.0503, "2017-04-18"),
    ("S06", "Antofagasta",  "Tienda mediana", "Antofagasta",  "Antofagasta",   -23.6509, -70.3975, "2020-02-03"),
    ("S07", "Viña del Mar", "Tienda mediana", "Viña del Mar", "Valparaíso",    -33.0245, -71.5518, "2018-09-27"),
    ("S08", "Temuco",       "Express",        "Temuco",       "La Araucanía",  -38.7359, -72.5904, "2021-06-15"),
], columns=["cod_sucursal", "sucursal", "formato", "comuna", "region", "latitud", "longitud", "fecha_apertura"])

sucursales_v2 = sucursales_v1.copy()
sucursales_v2.loc[sucursales_v2["cod_sucursal"] == "S03", "formato"] = "Tienda grande"    # remodelación Maipú
sucursales_v2.loc[sucursales_v2["cod_sucursal"] == "S08", "formato"] = "Tienda mediana"   # ampliación Temuco

sucursales_v1.to_csv(DIR_M1 / "sucursales.csv", index=False)
sucursales_v2.to_csv(DIR_M2 / "sucursales.csv", index=False)
sucursales_v2

,cod_sucursal,sucursal,formato,comuna,region,latitud,longitud,fecha_apertura
0,S01,Providencia,Tienda grande,Providencia,Metropolitana,-33.4255,-70.6114,2015-03-10
1,S02,Las Condes,Tienda grande,Las Condes,Metropolitana,-33.4170,-70.5500,2016-08-22
2,S03,Maipú,Tienda grande,Maipú,Metropolitana,-33.5110,-70.7580,2018-05-14
3,S04,Ñuñoa,Tienda mediana,Ñuñoa,Metropolitana,-33.4569,-70.5973,2019-11-04
4,S05,Concepción,Tienda grande,Concepción,Biobío,-36.8270,-73.0503,2017-04-18
5,S06,Antofagasta,Tienda mediana,Antofagasta,Antofagasta,-23.6509,-70.3975,2020-02-03
6,S07,Viña del Mar,Tienda mediana,Viña del Mar,Valparaíso,-33.0245,-71.5518,2018-09-27
7,S08,Temuco,Tienda mediana,Temuco,La Araucanía,-38.7359,-72.5904,2021-06-15


## 3. Maestro de clientes (v1 y v2)
RUT sintéticos con dígito verificador válido (misma función del 2.3.5), más **nombre, correo, teléfono, segmento, comuna y región**. El nombre, el correo y el teléfono existen solo para demostrar que el pipeline **seudonimiza y descarta** datos personales antes de Gold (Ley 21.719).

La v1 trae un **1% de clientes duplicados** (mismo RUT, correo con otra grafía): regla de unicidad del pipeline.

**Cambios de la v2:**

| Cambio | % aprox. | SCD sugerido | Por qué |
|---|---|---|---|
| Cambio de `segmento` | 6% | Tipo 2 | Se quiere medir migración entre segmentos y atribuir cada venta al segmento vigente en su fecha |
| Cambio de `comuna`/`region` (mudanza) | 2% | Tipo 2 | Análisis geográfico de clientes en el tiempo |
| Corrección de `email`/`telefono` | 3% | Tipo 1 | Corrección de contacto: la historia no tiene valor analítico |
| Clientes nuevos | 1.500 | Inserción | Registros posteriores a la carga inicial |

In [4]:
def dv_rut(n: int) -> str:
    s, m = 0, 2
    while n:
        s, n, m = s + (n % 10) * m, n // 10, 2 if m == 7 else m + 1
    r = 11 - s % 11
    return "0" if r == 11 else "K" if r == 10 else str(r)

def formatear_rut(n: int) -> str:
    return f"{n:,}".replace(",", ".") + "-" + dv_rut(int(n))

def sin_tildes(t: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")

NOMBRES   = ["Ana", "Camila", "Valentina", "Javiera", "Catalina", "Fernanda", "Constanza", "Daniela", "Sofía", "Isidora",
             "Benjamín", "Matías", "Vicente", "Martín", "Joaquín", "Tomás", "Diego", "Felipe", "Sebastián", "Cristóbal"]
APELLIDOS = ["González", "Muñoz", "Rojas", "Díaz", "Pérez", "Soto", "Contreras", "Silva", "Martínez", "Sepúlveda",
             "Morales", "Rodríguez", "López", "Fuentes", "Hernández", "Torres", "Araya", "Flores", "Espinoza", "Valenzuela"]
# comunas de residencia: las de las sucursales + otras de las mismas regiones
COMUNAS = pd.DataFrame([
    ("Providencia", "Metropolitana"), ("Las Condes", "Metropolitana"), ("Maipú", "Metropolitana"),
    ("Ñuñoa", "Metropolitana"), ("Santiago", "Metropolitana"), ("La Florida", "Metropolitana"),
    ("Puente Alto", "Metropolitana"), ("Concepción", "Biobío"), ("Talcahuano", "Biobío"),
    ("Antofagasta", "Antofagasta"), ("Calama", "Antofagasta"), ("Viña del Mar", "Valparaíso"),
    ("Valparaíso", "Valparaíso"), ("Temuco", "La Araucanía"), ("Padre Las Casas", "La Araucanía"),
], columns=["comuna", "region"])
PESO_COMUNA = np.array([10, 9, 9, 8, 8, 7, 7, 8, 4, 7, 3, 7, 4, 6, 3], dtype=float); PESO_COMUNA /= PESO_COMUNA.sum()
SEGMENTOS = ["Ocasional", "Frecuente", "Premium"]

def generar_clientes(n, fecha_min, fecha_max, excluir=frozenset()):
    numeros = set()
    while len(numeros) < n:
        for x in rng.integers(6_000_000, 26_000_000, n):
            if x not in excluir and len(numeros) < n:
                numeros.add(int(x))
    numeros = np.array(sorted(numeros))
    nom = rng.choice(NOMBRES, n); ape = rng.choice(APELLIDOS, n)
    com = COMUNAS.iloc[rng.choice(len(COMUNAS), n, p=PESO_COMUNA)].reset_index(drop=True)
    dias = (pd.Timestamp(fecha_max) - pd.Timestamp(fecha_min)).days
    return pd.DataFrame({
        "rut_cliente": [formatear_rut(x) for x in numeros],
        "nombre": [f"{a} {b}" for a, b in zip(nom, ape)],
        "email": [f"{sin_tildes(a).lower()}.{sin_tildes(b).lower()}{x % 1000}@correo.cl" for a, b, x in zip(nom, ape, numeros)],
        "telefono": [f"+569{rng.integers(10_000_000, 99_999_999)}" for _ in range(n)],
        "segmento": rng.choice(SEGMENTOS, n, p=[.60, .30, .10]),
        "comuna": com["comuna"], "region": com["region"],
        "fecha_registro": (pd.Timestamp(fecha_min) + pd.to_timedelta(rng.integers(0, dias, n), unit="D")).strftime("%Y-%m-%d"),
    }), set(numeros)

clientes_v1, nums_v1 = generar_clientes(N_CLIENTES, "2019-01-01", "2024-12-31")

# --- v2: cambios sobre los mismos clientes + clientes nuevos ---
clientes_v2 = clientes_v1.copy()
def idx(p): return clientes_v2.index[rng.choice(len(clientes_v2), int(len(clientes_v2) * p), replace=False)]

i_seg = idx(.06)                                                    # SCD2: segmento
clientes_v2.loc[i_seg, "segmento"] = [rng.choice([s for s in SEGMENTOS if s != actual])
                                      for actual in clientes_v2.loc[i_seg, "segmento"]]
i_mud = idx(.02)                                                    # SCD2: mudanza
nueva = COMUNAS.iloc[rng.choice(len(COMUNAS), len(i_mud), p=PESO_COMUNA)]
clientes_v2.loc[i_mud, "comuna"] = nueva["comuna"].to_numpy()
clientes_v2.loc[i_mud, "region"] = nueva["region"].to_numpy()
i_cor = idx(.03)                                                    # SCD1: corrección de contacto
clientes_v2.loc[i_cor, "email"] = clientes_v2.loc[i_cor, "email"].str.replace("@correo.cl", "@mail.cl", regex=False)
clientes_v2.loc[i_cor, "telefono"] = [f"+569{rng.integers(10_000_000, 99_999_999)}" for _ in i_cor]

nuevos, _ = generar_clientes(N_CLIENTES_NUEVOS, "2025-01-01", FECHA_MAESTROS2, excluir=nums_v1)
clientes_v2 = pd.concat([clientes_v2, nuevos], ignore_index=True)

# --- problema de calidad en v1: 1% duplicado con correo en otra grafía (unicidad) ---
dups = clientes_v1.sample(frac=.01, random_state=SEMILLA).copy()
dups["email"] = "  " + dups["email"].str.upper()
clientes_v1_sucio = pd.concat([clientes_v1, dups]).sort_index(kind="stable")

clientes_v1_sucio.to_csv(DIR_M1 / "clientes.csv", index=False)
clientes_v2.to_csv(DIR_M2 / "clientes.csv", index=False)
print(f"clientes v1: {len(clientes_v1_sucio):,} filas ({len(dups)} duplicados) | v2: {len(clientes_v2):,} filas")
print(f"cambios v2 → segmento: {len(i_seg)} · mudanzas: {len(i_mud)} · correcciones de contacto: {len(i_cor)} · nuevos: {len(nuevos)}")
clientes_v2.head()

clientes v1: 30,300 filas (300 duplicados) | v2: 31,500 filas
cambios v2 → segmento: 1800 · mudanzas: 600 · correcciones de contacto: 900 · nuevos: 1500


,rut_cliente,nombre,email,telefono,segmento,comuna,region,fecha_registro
0,6.000.017-4,Javiera Torres,javiera.torres17@correo.cl,+56913283861,Ocasional,Puente Alto,Metropolitana,2021-08-31
1,6.000.263-0,Javiera Sepúlveda,javiera.sepulveda263@correo.cl,+56969309682,Ocasional,Padre Las Casas,La Araucanía,2019-06-06
2,6.000.858-2,Matías Espinoza,matias.espinoza858@correo.cl,+56987620313,Frecuente,Santiago,Metropolitana,2019-08-30
3,6.002.498-7,Sebastián Pérez,sebastian.perez498@correo.cl,+56989242619,Ocasional,Viña del Mar,Valparaíso,2021-10-18
4,6.003.956-9,Catalina Muñoz,catalina.munoz956@correo.cl,+56918696619,Frecuente,Providencia,Metropolitana,2021-07-15


## 4. Maestro de campañas
Dimensión **Promoción/Campaña** (observación de la retroalimentación del EP1). Cada campaña tiene vigencia, porcentaje de descuento y alcance por categoría y canal. Si una línea califica para más de una campaña, se aplica la de **mayor descuento**. No todas las líneas que califican usan la campaña (70%).

In [5]:
campanas = pd.DataFrame([
    ("CMP-01", "Liquidación de Verano", "Liquidación",   0.15, "2025-01-02", "2025-01-26", "Todas",                 "Todos"),
    ("CMP-02", "Vuelta a Clases",       "Temporada",     0.10, "2025-02-15", "2025-03-15", "Tecnología|Accesorios", "Todos"),
    ("CMP-03", "Semana App",            "Canal digital", 0.08, "2025-02-03", "2025-02-09", "Todas",                 "App"),
    ("CMP-04", "Gaming Fest",           "Categoría",     0.20, "2025-03-20", "2025-03-31", "Gaming",                "Todos"),
], columns=["cod_campana", "campana", "tipo_campana", "pct_descuento", "fecha_inicio", "fecha_fin",
            "categorias_aplica", "canal_aplica"])
for d in (DIR_M1, DIR_M2):
    campanas.to_csv(d / "campanas.csv", index=False)
campanas

,cod_campana,campana,tipo_campana,pct_descuento,fecha_inicio,fecha_fin,categorias_aplica,canal_aplica
0,CMP-01,Liquidación de Verano,Liquidación,0.15,2025-01-02,2025-01-26,Todas,Todos
1,CMP-02,Vuelta a Clases,Temporada,0.10,2025-02-15,2025-03-15,Tecnología|Accesorios,Todos
2,CMP-03,Semana App,Canal digital,0.08,2025-02-03,2025-02-09,Todas,App
3,CMP-04,Gaming Fest,Categoría,0.20,2025-03-20,2025-03-31,Gaming,Todos


## 5. Ventas: boletas con líneas
Se generan **boletas** (fecha, sucursal, canal, cliente) y cada una tiene de 1 a 3 **líneas** (producto, cantidad). Así el grano del hecho —*una fila por línea de boleta*— queda identificado por `nro_boleta` + `nro_linea` (dimensión degenerada).

Reglas de negocio de la fuente:
- Fines de semana venden 40% más (igual que el 2.3.5).
- En Tienda física el 15% de las boletas no identifica cliente (`rut_cliente` vacío); en E-commerce y App siempre hay cliente.
- El `precio_unitario` y el catálogo disponible dependen de la versión del maestro vigente en la fecha (v1 antes de `FECHA_MAESTROS2`, v2 desde esa fecha).
- `monto` = precio_unitario × cantidad (**monto bruto**); `descuento` según campaña.
- `estado` = `DEVUELTA` en el 4% de las líneas (devolución de la línea completa).

In [6]:
SUCURSALES = sucursales_v1["sucursal"].to_numpy()
PESO_SUC   = [.18, .16, .14, .12, .12, .10, .10, .08]
CANALES    = ["Tienda física", "E-commerce", "App"]
fechas     = pd.date_range(FECHA_INICIO, periods=DIAS, freq="D")
corte      = pd.Timestamp(FECHA_MAESTROS2)

# --- boletas ---
n_bol = int(N_LINEAS / 1.6)
peso_dia = np.where(fechas.dayofweek >= 5, 1.4, 1.0); peso_dia /= peso_dia.sum()
bol = pd.DataFrame({"fecha": fechas[np.sort(rng.choice(DIAS, n_bol, p=peso_dia))]})
bol["nro_boleta"] = np.arange(1, n_bol + 1)
bol["sucursal"] = rng.choice(SUCURSALES, n_bol, p=PESO_SUC)
bol["canal"] = rng.choice(CANALES, n_bol, p=[.55, .35, .10])
ruts_v1 = clientes_v1["rut_cliente"].to_numpy()
ruts_v2 = clientes_v2["rut_cliente"].to_numpy()
desde_v2 = (bol["fecha"] >= corte).to_numpy()
bol["rut_cliente"] = np.where(desde_v2, rng.choice(ruts_v2, n_bol), rng.choice(ruts_v1, n_bol))
anonima = (bol["canal"].eq("Tienda física") & (rng.random(n_bol) < .15)).to_numpy()
bol.loc[anonima, "rut_cliente"] = ""

# --- líneas ---
n_lin = rng.choice([1, 2, 3], n_bol, p=[.55, .30, .15])
lin = bol.loc[bol.index.repeat(n_lin)].reset_index(drop=True)
lin["nro_linea"] = lin.groupby("nro_boleta").cumcount() + 1
es_v2 = (lin["fecha"] >= corte).to_numpy()
skus_v1, skus_v2 = productos_v1["sku"].to_numpy(), productos_v2["sku"].to_numpy()
lin["sku"] = np.where(es_v2, rng.choice(skus_v2, len(lin)), rng.choice(skus_v1, len(lin)))
lin["cantidad"] = rng.choice([1, 2, 3], len(lin), p=[.70, .20, .10])

# precio y categoría según la versión vigente en la fecha de la venta
p1 = productos_v1.set_index("sku"); p2 = productos_v2.set_index("sku")
lin["precio_unitario"] = np.where(es_v2, lin["sku"].map(p2["precio_lista"]), lin["sku"].map(p1["precio_lista"])).astype("int64")
categoria = np.where(es_v2, lin["sku"].map(p2["categoria"]), lin["sku"].map(p1["categoria"]))
lin["monto"] = lin["precio_unitario"] * lin["cantidad"]

# --- campañas: gana la de mayor descuento (se recorren de menor a mayor %) ---
lin["cod_campana"] = ""; pct = np.zeros(len(lin))
for c in campanas.sort_values("pct_descuento").itertuples():
    aplica = lin["fecha"].between(c.fecha_inicio, c.fecha_fin).to_numpy().copy()
    if c.categorias_aplica != "Todas":
        aplica &= np.isin(categoria, c.categorias_aplica.split("|"))
    if c.canal_aplica != "Todos":
        aplica &= lin["canal"].eq(c.canal_aplica).to_numpy()
    aplica &= rng.random(len(lin)) < .70
    lin.loc[aplica, "cod_campana"] = c.cod_campana; pct[aplica] = c.pct_descuento
lin["descuento"] = np.round(lin["monto"] * pct).astype("int64")
lin["estado"] = rng.choice(["OK", "DEVUELTA"], len(lin), p=[.96, .04])
lin["id_venta"] = np.arange(1, len(lin) + 1)
lin["fecha"] = lin["fecha"].dt.strftime("%Y-%m-%d")

COLS_VENTAS = ["id_venta", "nro_boleta", "nro_linea", "fecha", "sku", "sucursal", "canal", "cantidad",
               "precio_unitario", "monto", "descuento", "cod_campana", "estado", "rut_cliente"]
limpias = lin[COLS_VENTAS].astype(str)
print(f"{n_bol:,} boletas · {len(limpias):,} líneas · {limpias['cod_campana'].ne('').mean():.1%} con campaña · "
      f"{(limpias['rut_cliente'] == '').mean():.1%} sin cliente identificado")
limpias.head()

C:\Users\pedro\AppData\Local\Temp\ipykernel_13008\4229335569.py:9: Pandas4Warning: DatetimeIndex.dayofweek is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  peso_dia = np.where(fechas.dayofweek >= 5, 1.4, 1.0); peso_dia /= peso_dia.sum()


312,500 boletas · 499,751 líneas · 40.5% con campaña · 8.3% sin cliente identificado


,id_venta,nro_boleta,nro_linea,fecha,sku,sucursal,canal,cantidad,precio_unitario,monto,descuento,cod_campana,estado,rut_cliente
0,1,1,1,2025-01-01,CE-004,Temuco,Tienda física,3,349000,1047000,0,,OK,24.322.633-3
1,2,1,2,2025-01-01,MI-016,Temuco,Tienda física,1,89000,89000,0,,OK,24.322.633-3
2,3,2,1,2025-01-01,CO-015,Concepción,Tienda física,1,499000,499000,0,,OK,8.625.105-1
3,4,3,1,2025-01-01,TV-003,Maipú,Tienda física,1,459000,459000,0,,OK,19.495.942-7
4,5,4,1,2025-01-01,PA-010,Viña del Mar,E-commerce,2,59000,118000,0,,OK,9.029.167-K


## 6. Inyección de problemas de calidad
Se mantienen los problemas del 2.3.5 (mismos porcentajes) y se agregan dos propios de la extensión. Cada uno se asocia a una dimensión de calidad DAMA y a la regla que el pipeline deberá aplicar.

| Problema | % aprox. | Dimensión DAMA | Origen |
|---|---|---|---|
| SKU en minúsculas / con espacios | 5% | Consistencia | 2.3.5 |
| Fecha con `/` | 3% | Validez | 2.3.5 |
| Canal con otra grafía (`Tienda fisica`, `ecommerce`, `APP`) | 4% | Consistencia | 2.3.5 |
| Sucursal vacía | 0,8% | Completitud | 2.3.5 |
| Monto negativo | 0,4% | Validez | 2.3.5 |
| Monto no numérico (`N/A`) | 0,1% | Validez | 2.3.5 |
| SKU inexistente (`XX-999`) | 0,2% | Consistencia | 2.3.5 |
| Fecha imposible (`2025-02-30`) | 0,1% | Validez | 2.3.5 |
| Línea duplicada (mismo `id_venta`) | 1% | Unicidad | 2.3.5 |
| RUT sin puntos (`12345678-5`) | 2% | Consistencia | Extensión EP3 |
| Campaña inexistente (`CMP-99`) | 0,1% | Consistencia | Extensión EP3 |

In [7]:
def ensuciar(df):
    df = df.copy(); n = len(df); registro = {}
    def pos(nombre, p):
        i = df.index[rng.choice(n, size=max(int(n * p), 1), replace=False)]
        registro[nombre] = len(i); return i
    i = pos("sku_minusculas_espacios", .05); df.loc[i, "sku"] = df.loc[i, "sku"].str.lower() + " "
    i = pos("fecha_con_slash", .03);         df.loc[i, "fecha"] = df.loc[i, "fecha"].str.replace("-", "/")
    i = pos("canal_otra_grafia", .04)
    df.loc[i, "canal"] = df.loc[i, "canal"].map({"Tienda física": "Tienda fisica", "E-commerce": "ecommerce", "App": "APP"})
    i = pos("sucursal_vacia", .008);         df.loc[i, "sucursal"] = ""
    i = pos("monto_negativo", .004);         df.loc[i, "monto"] = "-" + df.loc[i, "monto"]
    i = pos("monto_no_numerico", .001);      df.loc[i, "monto"] = "N/A"
    i = pos("sku_inexistente", .002);        df.loc[i, "sku"] = "XX-999"
    i = pos("fecha_imposible", .001);        df.loc[i, "fecha"] = "2025-02-30"
    con_rut = df.index[df["rut_cliente"] != ""]
    i = con_rut[rng.choice(len(con_rut), int(n * .02), replace=False)]; registro["rut_sin_puntos"] = len(i)
    df.loc[i, "rut_cliente"] = df.loc[i, "rut_cliente"].str.replace(".", "", regex=False)
    i = pos("campana_inexistente", .001);    df.loc[i, "cod_campana"] = "CMP-99"
    duplicados = df.sample(frac=.01, random_state=SEMILLA); registro["linea_duplicada"] = len(duplicados)
    return pd.concat([df, duplicados]).sort_index(kind="stable"), registro

sucias, problemas = ensuciar(limpias)
pd.Series(problemas, name="filas afectadas").to_frame()

,filas afectadas
sku_minusculas_espacios,24987
fecha_con_slash,14992
canal_otra_grafia,19990
sucursal_vacia,3998
monto_negativo,1999
monto_no_numerico,499
sku_inexistente,999
fecha_imposible,499
rut_sin_puntos,9995
campana_inexistente,499


## 7. Escritura particionada (1 archivo por día)
La partición se toma de la fecha **original** de la venta, así una fecha mal escrita (`2025/01/05` o `2025-02-30`) queda dentro del archivo de su día, igual que en el 2.3.5.

In [8]:
t0 = time.perf_counter()
sucias = sucias.assign(_particion=lin.loc[sucias.index, "fecha"].to_numpy())
for particion, g in sucias.groupby("_particion"):
    g.drop(columns="_particion").to_csv(DIR_VENT / f"ventas_{particion}.csv", index=False)
archivos = sorted(DIR_VENT.glob("ventas_*.csv"))
mb = sum(a.stat().st_size for a in archivos) / 1e6
print(f"{len(sucias):,} filas en {len(archivos)} archivos ({mb:.1f} MB) · {time.perf_counter() - t0:.1f}s")
pd.read_csv(archivos[0], dtype=str, keep_default_na=False).head(8)

504,749 filas en 90 archivos (47.7 MB) · 1.5s


,id_venta,nro_boleta,nro_linea,fecha,sku,sucursal,canal,cantidad,precio_unitario,monto,descuento,cod_campana,estado,rut_cliente
0,1,1,1,2025-01-01,CE-004,Temuco,Tienda física,3,349000,1047000,0,,OK,24.322.633-3
1,2,1,2,2025-01-01,MI-016,Temuco,Tienda física,1,89000,89000,0,,OK,24.322.633-3
2,3,2,1,2025-01-01,CO-015,Concepción,Tienda física,1,499000,499000,0,,OK,8.625.105-1
3,4,3,1,2025-01-01,TV-003,Maipú,Tienda física,1,459000,459000,0,,OK,19.495.942-7
4,5,4,1,2025-01-01,PA-010,Viña del Mar,E-commerce,2,59000,118000,0,,OK,9.029.167-K
5,6,5,1,2025-01-01,NB-001,Viña del Mar,E-commerce,1,799000,799000,0,,OK,18.875.167-9
6,7,5,2,2025-01-01,MI-016,Viña del Mar,E-commerce,1,89000,89000,0,,OK,18.875.167-9
7,8,5,3,2025-01-01,CE-004,Viña del Mar,E-commerce,1,349000,349000,0,,OK,18.875.167-9


## 7.1 Inventario diario por producto y sucursal
Fuente del hecho **FACT_INVENTARIO_DIARIO**: una foto del stock al cierre de cada día, por sucursal y SKU (un archivo por día, igual que las ventas).
Se genera **después** de las ventas, por lo que las ventas no cambian con la semilla.

Reglas de la simulación, coherentes con las ventas generadas:
- Productos disponibles según el maestro vigente: 16 SKU hasta el 28-02 y 17 desde el 01-03 (GA-017 entra con stock de lanzamiento).
- `stock_minimo` = 4 días de venta promedio de ese SKU en esa sucursal; `stock_maximo` = 3 × `stock_minimo`.
- Stock de cierre = stock del día anterior − unidades vendidas + unidades devueltas + reposición.
- Si el cierre queda bajo el mínimo, al día siguiente llega reposición hasta el máximo.

| Problema inyectado | % aprox. | Dimensión DAMA |
|---|---|---|
| SKU en minúsculas / con espacios | 2% | Consistencia |
| `cantidad_stock` vacía | 0,3% | Completitud |
| Fila duplicada (misma fecha, sucursal y SKU) | 0,5% | Unicidad |

In [ ]:
COLS_INV = ["fecha", "cod_sucursal", "sku", "cantidad_stock", "stock_minimo"]
cod_suc = sucursales_v1.set_index("sucursal")["cod_sucursal"]

# movimientos diarios a partir de las ventas limpias (antes de ensuciar)
mov = (lin.assign(cod_sucursal=lin["sucursal"].map(cod_suc),
                  vendidas=lin["cantidad"],
                  devueltas=np.where(lin["estado"].eq("DEVUELTA"), lin["cantidad"], 0))
          .groupby(["fecha", "cod_sucursal", "sku"], as_index=False)[["vendidas", "devueltas"]].sum())

# grilla: cada día × sucursal × SKU disponible en el maestro vigente
dias_txt = pd.date_range(FECHA_INICIO, periods=DIAS, freq="D").strftime("%Y-%m-%d")
grilla = pd.DataFrame([(d, s, k) for d in dias_txt
                       for s in sucursales_v1["cod_sucursal"]
                       for k in (skus_v2 if d >= FECHA_MAESTROS2 else skus_v1)],
                      columns=["fecha", "cod_sucursal", "sku"])
grilla = grilla.merge(mov, on=["fecha", "cod_sucursal", "sku"], how="left").fillna({"vendidas": 0, "devueltas": 0})

# simulación del stock por sucursal y SKU
filas_inv = []
for (suc, sku), g in grilla.groupby(["cod_sucursal", "sku"], sort=True):
    g = g.sort_values("fecha")
    minimo = max(int(np.ceil(g["vendidas"].mean() * 4)), 5)
    maximo = minimo * 3
    stock = int(maximo if sku not in skus_v1 else rng.integers(int(minimo * 1.5), maximo + 1))
    pendiente = 0
    for r in g.itertuples(index=False):
        stock = max(stock + pendiente - int(r.vendidas) + int(r.devueltas), 0)
        pendiente = maximo - stock if stock < minimo else 0
        filas_inv.append((r.fecha, suc, sku, stock, minimo))
inventario = pd.DataFrame(filas_inv, columns=COLS_INV).sort_values(["fecha", "cod_sucursal", "sku"]).reset_index(drop=True)

# problemas de calidad
inv_sucio = inventario.astype(str)
n = len(inv_sucio); problemas_inv = {}
i = rng.choice(n, int(n * .02), replace=False);  problemas_inv["sku_minusculas_espacios"] = len(i)
inv_sucio.loc[i, "sku"] = inv_sucio.loc[i, "sku"].str.lower() + " "
i = rng.choice(n, int(n * .003), replace=False); problemas_inv["cantidad_stock_vacia"] = len(i)
inv_sucio.loc[i, "cantidad_stock"] = ""
dup_inv = inv_sucio.sample(frac=.005, random_state=SEMILLA); problemas_inv["fila_duplicada"] = len(dup_inv)
inv_sucio = pd.concat([inv_sucio, dup_inv]).sort_index(kind="stable")

for fecha_inv, g in inv_sucio.groupby("fecha"):
    g.to_csv(DIR_INV / f"inventario_{fecha_inv}.csv", index=False)
archivos_inv = sorted(DIR_INV.glob("inventario_*.csv"))
print(f"Inventario: {len(inventario):,} filas limpias · {len(inv_sucio):,} filas en {len(archivos_inv)} archivos")
print(f"Bajo stock mínimo al cierre: {(inventario['cantidad_stock'] < inventario['stock_minimo']).mean():.1%} de las filas")
pd.Series(problemas_inv, name="filas afectadas").to_frame()

## 8. Manifiesto de la fuente (trazabilidad)
`manifest.json` registra parámetros, semilla, conteos, problemas inyectados y la huella **SHA-256** de cada archivo. Permite verificar que el pipeline procesó exactamente esta fuente y reproducirla.

In [9]:
def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

todos = sorted(p for p in FUENTE.rglob("*") if p.is_file())
manifest = {
    "generado_en": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    "parametros": {"N_LINEAS": N_LINEAS, "DIAS": DIAS, "FECHA_INICIO": FECHA_INICIO,
                   "FECHA_MAESTROS2": FECHA_MAESTROS2, "N_CLIENTES": N_CLIENTES,
                   "N_CLIENTES_NUEVOS": N_CLIENTES_NUEVOS, "SEMILLA": SEMILLA},
    "conteos": {"boletas": int(n_bol), "lineas_limpias": len(limpias), "filas_fuente": len(sucias),
                "archivos_ventas": len(archivos), "productos_v1": len(productos_v1), "productos_v2": len(productos_v2),
                "clientes_v1": len(clientes_v1_sucio), "clientes_v2": len(clientes_v2),
                "sucursales": len(sucursales_v1), "campanas": len(campanas),
                "filas_inventario": len(inv_sucio), "archivos_inventario": len(archivos_inv)},
    "problemas_inyectados": problemas,
    "problemas_inyectados_inventario": problemas_inv,
    "archivos": {str(p.relative_to(FUENTE)).replace("\\", "/"): sha256(p) for p in todos},
}
(FUENTE / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps({k: manifest[k] for k in ("parametros", "conteos")}, ensure_ascii=False, indent=1))

{
 "parametros": {
  "N_LINEAS": 500000,
  "DIAS": 90,
  "FECHA_INICIO": "2025-01-01",
  "FECHA_MAESTROS2": "2025-03-01",
  "N_CLIENTES": 30000,
  "N_CLIENTES_NUEVOS": 1500,
  "SEMILLA": 42
 },
 "conteos": {
  "boletas": 312500,
  "lineas_limpias": 499751,
  "filas_fuente": 504749,
  "archivos_ventas": 90,
  "productos_v1": 16,
  "productos_v2": 17,
  "clientes_v1": 30300,
  "clientes_v2": 31500,
  "sucursales": 8,
  "campanas": 4
 }
}


## 9. Verificación: cambios entre v1 y v2 (insumo de los SCD)
Comparamos ambas entregas de maestros para confirmar qué atributos cambiaron. Esta tabla es la evidencia que justifica el tipo de SCD de cada atributo en el informe.

In [10]:
def comparar(v1, v2, clave, atributos, scd):
    m = v1.drop_duplicates(clave).merge(v2, on=clave, how="outer", suffixes=("_v1", "_v2"), indicator=True)
    filas = [{"atributo": "(alta nueva)", "registros": int((m["_merge"] == "right_only").sum()), "scd": "Inserción"}]
    for a in atributos:
        ambos = m[m["_merge"] == "both"]
        filas.append({"atributo": a, "registros": int((ambos[f"{a}_v1"] != ambos[f"{a}_v2"]).sum()), "scd": scd[a]})
    return pd.DataFrame(filas)

resumen = pd.concat([
    comparar(productos_v1, productos_v2, "sku", ["precio_lista", "categoria", "marca"],
             {"precio_lista": "Tipo 2", "categoria": "Tipo 2", "marca": "Tipo 1"}).assign(maestro="productos"),
    comparar(sucursales_v1, sucursales_v2, "cod_sucursal", ["formato"], {"formato": "Tipo 3"}).assign(maestro="sucursales"),
    comparar(clientes_v1, clientes_v2, "rut_cliente", ["segmento", "comuna", "email", "telefono"],
             {"segmento": "Tipo 2", "comuna": "Tipo 2", "email": "Tipo 1", "telefono": "Tipo 1"}).assign(maestro="clientes"),
])
resumen[["maestro", "atributo", "registros", "scd"]].reset_index(drop=True)

,maestro,atributo,registros,scd
0,productos,(alta nueva),1,Inserción
1,productos,precio_lista,3,Tipo 2
2,productos,categoria,1,Tipo 2
3,productos,marca,1,Tipo 1
4,sucursales,(alta nueva),0,Inserción
5,sucursales,formato,2,Tipo 3
6,clientes,(alta nueva),1500,Inserción
7,clientes,segmento,1800,Tipo 2
8,clientes,comuna,543,Tipo 2
9,clientes,email,900,Tipo 1


## 10. Diccionario de la fuente

**`data/fuente/ventas/ventas_AAAA-MM-DD.csv`** (una fila por línea de boleta)

| Columna | Descripción | Ejemplo |
|---|---|---|
| `id_venta` | Identificador único de la línea (clave de negocio) | 1834 |
| `nro_boleta` | Número de boleta (dimensión degenerada) | 1120 |
| `nro_linea` | Posición de la línea en la boleta | 2 |
| `fecha` | Fecha de la venta (AAAA-MM-DD) | 2025-01-05 |
| `sku` | Código de producto | NB-001 |
| `sucursal` | Sucursal que vende o despacha | Providencia |
| `canal` | Tienda física · E-commerce · App | App |
| `cantidad` | Unidades | 2 |
| `precio_unitario` | Precio de lista vigente en la fecha | 799000 |
| `monto` | Monto bruto = precio × cantidad | 1598000 |
| `descuento` | Descuento por campaña | 239700 |
| `cod_campana` | Campaña aplicada (vacío = sin campaña) | CMP-01 |
| `estado` | OK · DEVUELTA | OK |
| `rut_cliente` | RUT del cliente (dato personal; vacío = no identificado) | 12.345.678-5 |

**`data/fuente/maestros/v1|v2/`**: `productos.json`, `sucursales.csv`, `clientes.csv`, `campanas.csv` (columnas descritas en las secciones 1 a 4).

**`data/fuente/inventario/inventario_AAAA-MM-DD.csv`** (una fila por sucursal y SKU al cierre del día)

| Columna | Descripción | Ejemplo |
|---|---|---|
| `fecha` | Día de la foto de stock | 2025-01-05 |
| `cod_sucursal` | Código de sucursal | S01 |
| `sku` | Código de producto | NB-001 |
| `cantidad_stock` | Unidades en stock al cierre (semi-aditiva: no se suma entre días) | 154 |
| `stock_minimo` | Punto de reposición | 72 |

**`data/fuente/manifest.json`**: parámetros, conteos, problemas inyectados y SHA-256 de cada archivo.

---
*Siguiente notebook: `02_pipeline_medallion.ipynb` (Bronze → Silver → Gold).*